# Lab 2 — Aerial House Segmentation

**Before running anything:**
1. Go to **Runtime → Change runtime type → T4 GPU** → Save
2. Then run all cells top to bottom (Runtime → Run all)

Steps in this notebook:
1. Verify GPU
2. Clone repo
3. Install dependencies
4. Prepare dataset
5. Train model
6. Evaluate and visualise results

## 1 — Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

## 2 — Clone repo

In [ ]:
import os

if not os.path.exists('Lab2'):
    !git clone https://github.com/MalekDavid01/MyLab2.git Lab2

os.chdir('Lab2')
print('Working directory:', os.getcwd())

## 3 — Install dependencies

We pin `datasets<3.0.0` first because version 3.0 dropped support for custom dataset
loading scripts, which the aerial building dataset uses.

In [ ]:
# datasets v3 dropped support for custom loading scripts — pin to v2
!pip install -q 'datasets>=2.14.0,<3.0.0'
!pip install -q -r requirements.txt

## 4 — Create .env config

In [ ]:
env_content = """CHECKPOINT_DIR=checkpoints/best
IMG_SIZE=512
PORT=5000
SEGFORMER_MODEL=nvidia/mit-b0
EPOCHS=10
BATCH_SIZE=4
LEARNING_RATE=6e-5
NUM_WORKERS=2
SAM_CHECKPOINT_PATH=sam_vit_h_4b8939.pth
IOU_THRESHOLD=0.3
DATA_OUTPUT_DIR=data
MAX_SAMPLES=500
EVAL_OUTPUT_DIR=eval_outputs
VIZ_SAMPLES=8
"""

with open('.env', 'w') as f:
    f.write(env_content)

print('.env written')

## 5 — Prepare dataset

Downloads the aerial image dataset from HuggingFace and generates binary pixel masks
from polygon annotations.  
SAM is not used here (no checkpoint downloaded) — the polygon fallback produces
equivalent masks for this dataset.

In [ ]:
!python prepare_dataset.py

In [ ]:
import json
with open('data/manifest.json') as f:
    manifest = json.load(f)
print('train:', len(manifest['train']),
      ' val:', len(manifest['val']),
      ' test:', len(manifest['test']))

## 6 — Train model

Fine-tunes SegFormer (`nvidia/mit-b0`) for 10 epochs.  
Prints IoU and Dice score after each epoch.  
Best checkpoint saved to `checkpoints/best/`.

In [ ]:
!python train.py

## 7 — Evaluate on test set

In [ ]:
!python evaluate.py

In [ ]:
import json
with open('eval_outputs/metrics.json') as f:
    metrics = json.load(f)
print('Test set results:')
for k, v in metrics.items():
    print(f'  {k}: {v}')

## 8 — Visualise predictions

In [ ]:
from IPython.display import Image, display
print('Prediction grid (aerial image | ground truth | prediction):')
display(Image('eval_outputs/predictions_grid.png'))

In [ ]:
print('Training curves (loss + IoU/Dice over epochs):')
display(Image('eval_outputs/training_curves.png'))